# Solution E3 · Monotonic constraints ruin the model (or is it the order?)

**Type**: Find the bug

## Objective

A colleague adds `monotone_constraints=(-1, 0, 1, 0)` to a loan repayment model ("more income, more probability of repayment; more debt, less") and finds that the AUC collapses from 0.83 to 0.65: they conclude that "monotonic constraints do not work". Find the bug (the tuple follows a **different order** from the columns), fix it with a dictionary by name and
**check** that the model really is monotonic and that `income` has an effect again.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

import xgboost as xgb
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(0)
n = 4000
X = pd.DataFrame({"income": rng.normal(size=n), "debt": rng.normal(size=n), "age": rng.normal(size=n), "noise": rng.normal(size=n)})
y = rng.binomial(1, 1 / (1 + np.exp(-(1.5 * X["income"] - 1.0 * X["debt"] + 0.3 * X["age"])))).astype(int)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0)


def model(**kw):
    return xgb.XGBClassifier(n_estimators=150, learning_rate=0.1, max_depth=3, eval_metric="auc", n_jobs=-1, random_state=0, **kw)


def profile(m, X_ref, column, points=30):
    """Probability as one column varies (from the 5th to the 95th percentile), with the other columns at the median."""
    grid = np.linspace(X_ref[column].quantile(0.05), X_ref[column].quantile(0.95), points)
    base = pd.DataFrame([X_ref.median()] * points).reset_index(drop=True)
    base[column] = grid
    return m.predict_proba(base)[:, 1]

## Commented solution

In [ ]:
constraints = (-1, 0, 1, 0)            # debt (-), age, income (+), noise: 'I wrote them in the order I thought of the features'
free_model = model().fit(X_tr, y_tr)
bug_model = model(monotone_constraints=constraints).fit(X_tr, y_tr)
free_auc = roc_auc_score(y_te, free_model.predict_proba(X_te)[:, 1])
bug_auc = roc_auc_score(y_te, bug_model.predict_proba(X_te)[:, 1])
print(f"AUC without constraints {free_auc:.3f} | with the colleague's constraints {bug_auc:.3f} -> 'monotonic constraints ruin the model'")

**Where the bug is.** The tuple is **positional**: the first value goes to the first column of the DataFrame (`income`), not to `debt`. The colleague wrote the constraints in a different order from the columns, which are `income, debt, age, noise`, so `income` receives `-1` ("the probability cannot increase with income") and `age` receives `+1`.
Since income really has an increasing effect, every useful split is forbidden: the model cannot use the most important feature and ignores it. It is not a flaw of monotonic constraints.

In [ ]:
applied = dict(zip(X_tr.columns, constraints))
real_signs = {c: int(np.sign(np.corrcoef(X_tr[c], y_tr)[0, 1])) for c in ["income", "debt", "age"]}
wrong = sorted(c for c in ["income", "debt", "age"] if applied[c] != 0 and applied[c] != real_signs[c])
print("constraints applied:", applied)
print("real signs:", real_signs, "| constraints with the opposite sign:", wrong)

In [ ]:
ok_constraints = {"income": 1, "debt": -1}
ok_model = model(monotone_constraints=ok_constraints).fit(X_tr, y_tr)
ok_auc = roc_auc_score(y_te, ok_model.predict_proba(X_te)[:, 1])
print(f"AUC with the correct constraints {ok_auc:.3f} (free {free_auc:.3f})")

In [ ]:
def n_violations(m, column, sign=1):
    return int(((sign * np.diff(profile(m, X_te, column))) < -1e-9).sum())


def swing(m, column):
    p = profile(m, X_te, column)
    return float(p.max() - p.min())


bug_swing, ok_swing = swing(bug_model, "income"), swing(ok_model, "income")
free_violations = n_violations(free_model, "income")
monotone_ok = bool(n_violations(ok_model, "income", 1) == 0 and n_violations(ok_model, "debt", -1) == 0)
print(f"swing of the probability as income varies: colleague's model {bug_swing:.3f}, corrected {ok_swing:.3f}")
print(f"points (out of 29) where income makes the probability drop: free model {free_violations}, corrected {n_violations(ok_model, 'income')}")
print("the corrected model is monotonic for income (+) and debt (-):", monotone_ok)

## Check

In [ ]:
with verify():
    check(free_auc > 0.8 and bug_auc < free_auc - 0.1, "without constraints the AUC goes over 0.8; with the colleague's it loses more than 0.1", "don't modify the colleague's code")
    check(applied == {"income": -1, "debt": 0, "age": 1, "noise": 0}, "the tuple applied to the columns: income -1, debt 0, age +1, noise 0", "dict(zip(X.columns, tuple))")
    check(real_signs.get("income") == 1 and real_signs.get("debt") == -1 and real_signs.get("age") == 1, "real signs: income +, debt -, age +", "np.sign(np.corrcoef(X[c], y)[0, 1])")
    check(wrong == ["income"], "the only column with a constraint of the opposite sign to the real one is income (for age the sign matches by chance)", "applied constraint ≠ real sign, both different from 0")
    check(ok_constraints == {"income": 1, "debt": -1} and ok_auc is not None and ok_auc > free_auc - 0.01, "with the dictionary by name the AUC goes back to the free one (or exceeds it)", "monotone_constraints={'income': 1, 'debt': -1}")
    check(bug_swing is not None and bug_swing < 0.02 and ok_swing > 0.3, "in the colleague's model the probability is flat with respect to income (swing below 0.02); in the corrected one it changes by over 0.3", "profile(m, X_te, 'income').max() - .min()")
    check(free_violations is not None and free_violations > 0 and monotone_ok is True, "the free model has points where income makes the probability drop (spurious oscillations); the corrected one is monotonic for income (+) and debt (−)", "np.diff(profile(...))")
done()

## What you learned

- **`monotone_constraints` as a tuple is positional:** the i-th value goes to the i-th column and XGBoost does not check the names. With the wrong order the constraint is applied to another feature: here `income` receives `-1`, but the real effect is increasing, so the model cannot use it (its profile is flat) and the AUC drops from 0.83 to 0.65. The colleague blamed the constraints.
- **The dictionary by name is safer:** `{"income": 1, "debt": -1}` (XGBoost 1.6 accepts it) does not depend on the order of the columns, and a ColumnTransformer that shuffles them no longer breaks anything. In pipelines, where the output names change, the tuple is a time bomb.
- **A right constraint costs almost nothing:** the constrained AUC is equal to the free one (0.827 against 0.827). When the constraint is consistent with reality, it only removes the spurious oscillations and it is a gift for explainability (and for regulators, in credit/insurance).
- **Always verify the constraint, do not trust it:** with `profile` (one feature varying, the others at the median) you measure two things: the monotonicity violations (the free model has 5 out of 29 points, the corrected one 0 by construction) and the swing, that is how much the probability really changes. A wrong-sign constraint is not seen from the violations (a flat profile is monotonic!) but from the null swing. It is a test to put in the validation pipeline.

## Common pitfall

Passing `monotone_constraints` as a tuple after a preprocessing step that changes the order of the columns (ColumnTransformer, one-hot): the constraints shift position without errors. Use the dictionary by name or build the tuple from the column names.

## Interview question

<details><summary>What are monotonic constraints in XGBoost for and how do you verify them?</summary>

They force the prediction to be nondecreasing (or nonincreasing) with respect to a feature: useful when the domain requires it (credit, prices, insurance) and as a regularization. They cost little if the constraint is consistent with the data. I specify them by name (not by position) and verify them with a profile: I vary the feature keeping the others fixed and
check that the differences do not change sign.

</details>